# LeetCode 226 — Invert Binary Tree bằng Python

**Link đề:** https://leetcode.com/problems/invert-binary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def invertTree(self, root: TreeNode | None) -> TreeNode | None:
        ...
```

## Mục tiêu học
1. Hiểu **cây nhị phân** (binary tree): node, `left`, `right`, `root`, lá (leaf), cây con (subtree).
2. Biết LeetCode biểu diễn cây bằng list **theo tầng** (level order) như `[4,2,7,1,3,6,9]`, và tự viết hàm dựng/in cây.
3. Hiểu "lật cây" = **đổi chỗ con trái và con phải ở MỌI node** (ảnh gương).
4. Viết 4 lời giải: dựng cây gương mới → đệ quy (DFS) → BFS bằng queue → DFS bằng stack.
5. Trace call stack đệ quy, chứng minh bằng quy nạp, phân tích Time/Space theo chiều cao `h`.

## 1. Đề bài

Cho `root` của một cây nhị phân, hãy **lật** (invert) cây — tức tạo ảnh gương qua trục dọc — và trả về `root`.

```text
Input:        4                 Output:       4
            /   \                           /   \
           2     7                         7     2
          / \   / \                       / \   / \
         1   3 6   9                     9   6 3   1
```

| Input (level order) | Output |
|---|---|
| `[4,2,7,1,3,6,9]` | `[4,7,2,9,6,3,1]` |
| `[2,1,3]` | `[2,3,1]` |
| `[]` | `[]` |

**Constraints:** số node trong `[0, 100]`; `-100 <= Node.val <= 100`.

Lưu ý: "lật" **không** phải đổi giá trị, cũng **không** chỉ đổi hai con của root — mà đổi con trái/phải ở **mọi** node.

## 2. Nền tảng: `TreeNode` và thuật ngữ cây

Mỗi node có:
- `val`: giá trị,
- `left`: con trái (một `TreeNode` hoặc `None`),
- `right`: con phải.

Thuật ngữ:
- **root** (gốc): node trên cùng.
- **leaf** (lá): node không có con nào.
- **subtree** (cây con): một node cùng toàn bộ hậu duệ của nó — bản thân cũng là một cây nhị phân. Đây là lý do đệ quy hợp với cây.
- **height** (chiều cao) `h`: số node trên đường dài nhất từ root xuống lá.

In [1]:
from typing import Optional

class TreeNode:
    def __init__(self, val: int = 0, left: "Optional[TreeNode]" = None, right: "Optional[TreeNode]" = None):
        self.val = val
        self.left = left
        self.right = right

# Tự tay dựng cây [2, 1, 3]
root = TreeNode(2, TreeNode(1), TreeNode(3))
print("root:", root.val, "| left:", root.left.val, "| right:", root.right.val)
print("node 1 là lá?", root.left.left is None and root.left.right is None)

root: 2 | left: 1 | right: 3
node 1 là lá? True


### 2.1. Biểu diễn level order của LeetCode và `deque`

LeetCode ghi cây theo từng **tầng**, trái sang phải; ô `null` (Python: `None`) nghĩa là vị trí đó không có node.

Ví dụ `[1, None, 2, 3]`: root 1, con trái rỗng, con phải 2; node 2 có con trái 3.

Để dựng/đọc theo tầng, ta dùng **queue** `collections.deque`: `append` thêm cuối, `popleft` lấy đầu (FIFO — vào trước ra trước). Node nào được tạo trước thì được gán con trước.

In [2]:
from collections import deque

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:      # con trái
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:      # con phải
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_level_list(root: Optional[TreeNode]) -> list:
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:      # bỏ các None thừa ở cuối như LeetCode
        out.pop()
    return out

print(to_level_list(build_tree([4, 2, 7, 1, 3, 6, 9])))
print(to_level_list(build_tree([1, None, 2, 3])))
print(to_level_list(build_tree([])))

[4, 2, 7, 1, 3, 6, 9]
[1, None, 2, 3]
[]


### 2.2. In cây "nằm ngang" để nhìn bằng mắt

Hàm `draw` in cây xoay 90°: con **phải** ở trên, con **trái** ở dưới, thụt lề theo độ sâu. Hàm này dùng **đệ quy** — một hàm tự gọi chính nó trên cây con. Đọc kỹ: mỗi lời gọi chỉ lo **một** node, phần còn lại giao cho lời gọi con.

In [3]:
def draw(node: Optional[TreeNode], depth: int = 0) -> None:
    if node is None:
        return
    draw(node.right, depth + 1)
    print("      " * depth + str(node.val))
    draw(node.left, depth + 1)

draw(build_tree([4, 2, 7, 1, 3, 6, 9]))

            9
      7
            6
4
            3
      2
            1


Đọc hình: `4` là root (cột trái nhất). Phía trên `4` là nhánh **phải** (`7` với `9` trên, `6` dưới); phía dưới là nhánh **trái** (`2` với `3` trên, `1` dưới).

### 2.3. Đệ quy (recursion) — ví dụ khởi động: đếm số node

Hai phần của mọi hàm đệ quy trên cây:
1. **Trường hợp cơ sở** (base case): cây rỗng `None` → trả ngay.
2. **Bước đệ quy:** xử lý node hiện tại + gọi đệ quy cho `left` và `right`.

`count(node) = 0` nếu `node is None`, ngược lại `1 + count(left) + count(right)`.

In [4]:
def count_nodes(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def height(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

t = build_tree([4, 2, 7, 1, 3, 6, 9])
print("Số node:", count_nodes(t), "| chiều cao:", height(t))

Số node: 7 | chiều cao: 3


## 3. Quan sát: lật cây = đổi chỗ hai con ở **mọi** node

Thử chỉ đổi hai con của root:

In [5]:
t = build_tree([4, 2, 7, 1, 3, 6, 9])
t.left, t.right = t.right, t.left          # chỉ đổi ở root
print("Chỉ đổi ở root:", to_level_list(t), "| mong muốn: [4, 7, 2, 9, 6, 3, 1]")

Chỉ đổi ở root: [4, 7, 2, 6, 9, 1, 3] | mong muốn: [4, 7, 2, 9, 6, 3, 1]


Chưa đúng: `[4, 7, 2, 6, 9, 1, 3]` — hai cây con đã đổi vị trí nhưng **bên trong** mỗi cây con chưa được lật. Phải làm tiếp ở `7` và `2`, rồi ở các node con của chúng...

⇒ Định nghĩa đệ quy tự nhiên:

> **Lật cây gốc `node`** = đổi chỗ `node.left` và `node.right`, rồi **lật cây con trái** và **lật cây con phải**.

`a, b = b, a` là cách đổi chỗ trong Python: vế phải được tính xong (thành một tuple) trước khi gán.

## 4. Cách 1 — Baseline: dựng một cây gương **mới**

Cách nghĩ an toàn nhất cho người mới: không đụng vào cây cũ, tạo cây mới sao cho:
- `mirror(None) = None`
- `mirror(node)` = node mới cùng `val`, có `left = mirror(node.right)` và `right = mirror(node.left)`.

Đúng, nhưng tạo `n` node mới → bộ nhớ `O(n)` thêm. Đề cho phép sửa tại chỗ và trả về `root`, nên ta sẽ tối ưu ở cách 2.

In [6]:
def mirror_copy(node: Optional[TreeNode]) -> Optional[TreeNode]:
    if node is None:
        return None
    return TreeNode(node.val, mirror_copy(node.right), mirror_copy(node.left))

orig = build_tree([4, 2, 7, 1, 3, 6, 9])
m = mirror_copy(orig)
print("Cây gương mới:", to_level_list(m))
print("Cây gốc giữ nguyên:", to_level_list(orig))
print("root mới có phải root cũ?", m is orig)

Cây gương mới: [4, 7, 2, 9, 6, 3, 1]
Cây gốc giữ nguyên: [4, 2, 7, 1, 3, 6, 9]
root mới có phải root cũ? False


## 5. Cách 2 — Lời giải chính: đệ quy sửa tại chỗ (DFS)

Từng đoạn:

**Đoạn 1 — cơ sở**
```python
if root is None:
    return None
```

**Đoạn 2 — đổi chỗ hai con của node hiện tại**
```python
root.left, root.right = root.right, root.left
```

**Đoạn 3 — lật hai cây con (đã đổi chỗ, nhưng lật cái nào trước cũng được)**
```python
invert(root.left)
invert(root.right)
```

**Đoạn 4 — trả root** (đề yêu cầu trả root; root không đổi, chỉ cấu trúc bên dưới đổi)
```python
return root
```

Thứ tự "xử lý node → rồi mới đi xuống con" gọi là **preorder** (tiền thứ tự). Đổi chỗ sau khi đã lật hai con (**postorder**) cũng đúng.

In [7]:
def invert_recursive(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if root is None:
        return None
    root.left, root.right = root.right, root.left
    invert_recursive(root.left)
    invert_recursive(root.right)
    return root

for vals in [[4, 2, 7, 1, 3, 6, 9], [2, 1, 3], [], [1, 2], [1, None, 2, 3]]:
    print(f"{vals!s:<24} -> {to_level_list(invert_recursive(build_tree(vals)))}")

[4, 2, 7, 1, 3, 6, 9]    -> [4, 7, 2, 9, 6, 3, 1]
[2, 1, 3]                -> [2, 3, 1]
[]                       -> []
[1, 2]                   -> [1, None, 2]
[1, None, 2, 3]          -> [1, 2, None, None, 3]


### 5.1. Bản postorder một dòng

```python
root.left, root.right = invert(root.right), invert(root.left)
```
Vế phải gọi đệ quy lật hai cây con **trước**, rồi gán chéo. Gọn và cũng đúng.

In [8]:
def invert_postorder(root):
    if root is None:
        return None
    root.left, root.right = invert_postorder(root.right), invert_postorder(root.left)
    return root

t = build_tree([4, 2, 7, 1, 3, 6, 9])
print(to_level_list(invert_postorder(t)))
draw(t)

[4, 7, 2, 9, 6, 3, 1]
            1
      2
            3
4
            6
      7
            9


## 6. TRACE đệ quy — xem call stack

### 6.1. Bảng trace preorder trên `[4, 2, 7, 1, 3, 6, 9]`

Mỗi dòng là một lần "thăm" node (theo thứ tự preorder **sau khi** đã đổi con ở node cha):

| Thứ tự | Node | Con trái trước → sau | Con phải trước → sau | Gọi tiếp |
|---:|:-:|:-:|:-:|---|
| 1 | 4 | 2 → **7** | 7 → **2** | lật(7), rồi lật(2) |
| 2 | 7 | 6 → **9** | 9 → **6** | lật(9), lật(6) |
| 3 | 9 | None | None | lá, đổi None↔None |
| 4 | 6 | None | None | lá |
| 5 | 2 | 1 → **3** | 3 → **1** | lật(3), lật(1) |
| 6 | 3 | None | None | lá |
| 7 | 1 | None | None | lá |

Mỗi node được thăm đúng **một lần**; các lời gọi với `None` trả về ngay (không ghi trong bảng).

In [9]:
def trace_invert(root, depth=0):
    pad = "    " * depth
    if root is None:
        print(pad + "lật(None) -> cơ sở, trả None")
        return None
    l = root.left.val if root.left else None
    r = root.right.val if root.right else None
    root.left, root.right = root.right, root.left
    print(pad + f"lật({root.val}): đổi trái/phải ({l}, {r}) -> ({r}, {l})")
    trace_invert(root.left, depth + 1)
    trace_invert(root.right, depth + 1)
    return root

t = trace_invert(build_tree([4, 2, 7, 1, 3, 6, 9]))
print("Kết quả:", to_level_list(t))

lật(4): đổi trái/phải (2, 7) -> (7, 2)
    lật(7): đổi trái/phải (6, 9) -> (9, 6)
        lật(9): đổi trái/phải (None, None) -> (None, None)
            lật(None) -> cơ sở, trả None
            lật(None) -> cơ sở, trả None
        lật(6): đổi trái/phải (None, None) -> (None, None)
            lật(None) -> cơ sở, trả None
            lật(None) -> cơ sở, trả None
    lật(2): đổi trái/phải (1, 3) -> (3, 1)
        lật(3): đổi trái/phải (None, None) -> (None, None)
            lật(None) -> cơ sở, trả None
            lật(None) -> cơ sở, trả None
        lật(1): đổi trái/phải (None, None) -> (None, None)
            lật(None) -> cơ sở, trả None
            lật(None) -> cơ sở, trả None
Kết quả: [4, 7, 2, 9, 6, 3, 1]


Thụt lề = độ sâu của **call stack**: lúc sâu nhất có `h + 1` lời gọi đang chờ (`h` = chiều cao, +1 cho lời gọi với `None`). Đó là bộ nhớ mà đệ quy tốn.

### 6.2. Trace cây lệch (skewed) — trường hợp chiều cao lớn nhất

Cây `[1, 2, None, 3]` lệch hẳn sang trái. Sau khi lật sẽ lệch hẳn sang phải: `[1, None, 2, None, 3]`.

In [10]:
t = trace_invert(build_tree([1, 2, None, 3]))
print("Kết quả:", to_level_list(t))
draw(t)

lật(1): đổi trái/phải (2, None) -> (None, 2)
    lật(None) -> cơ sở, trả None
    lật(2): đổi trái/phải (3, None) -> (None, 3)
        lật(None) -> cơ sở, trả None
        lật(3): đổi trái/phải (None, None) -> (None, None)
            lật(None) -> cơ sở, trả None
            lật(None) -> cơ sở, trả None
Kết quả: [1, None, 2, None, 3]
            3
      2
1


## 7. Cách 3 — BFS (duyệt theo tầng) bằng queue

Không cần đệ quy: chỉ cần **thăm mọi node một lần** và đổi hai con của nó — thứ tự thăm không quan trọng.

BFS (Breadth-First Search) thăm theo tầng:
1. `queue = deque([root])`.
2. Lấy `node = queue.popleft()`, đổi hai con.
3. Đưa các con khác `None` vào cuối queue.
4. Lặp tới khi queue rỗng.

In [11]:
def invert_bfs(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if root is None:
        return None
    queue = deque([root])
    while queue:
        node = queue.popleft()
        node.left, node.right = node.right, node.left
        if node.left:
            queue.append(node.left)
        if node.right:
            queue.append(node.right)
    return root

def trace_bfs(root):
    queue = deque([root]) if root else deque()
    step = 1
    while queue:
        node = queue.popleft()
        node.left, node.right = node.right, node.left
        for child in (node.left, node.right):
            if child:
                queue.append(child)
        print(f"Bước {step}: thăm {node.val}, đổi con -> queue = {[n.val for n in queue]}")
        step += 1
    return root

print(to_level_list(trace_bfs(build_tree([4, 2, 7, 1, 3, 6, 9]))))

Bước 1: thăm 4, đổi con -> queue = [7, 2]
Bước 2: thăm 7, đổi con -> queue = [2, 9, 6]
Bước 3: thăm 2, đổi con -> queue = [9, 6, 3, 1]
Bước 4: thăm 9, đổi con -> queue = [6, 3, 1]
Bước 5: thăm 6, đổi con -> queue = [3, 1]
Bước 6: thăm 3, đổi con -> queue = [1]
Bước 7: thăm 1, đổi con -> queue = []
[4, 7, 2, 9, 6, 3, 1]


`if node.left:` — một đối tượng `TreeNode` được coi là `True`, `None` là `False`. Viết vậy tương đương `if node.left is not None:` (ở đây an toàn vì `TreeNode` không định nghĩa `__bool__`/`__len__`).

## 8. Cách 4 — DFS lặp bằng stack

Thay queue (FIFO) bằng **stack** (LIFO, dùng `list.append` / `list.pop()`) → thứ tự thăm thành theo chiều sâu. Kết quả cuối cùng vẫn giống, vì mỗi node vẫn được đổi đúng một lần.

Ưu điểm so với đệ quy: không bị giới hạn độ sâu đệ quy của Python (~1000) khi cây rất sâu.

In [12]:
def invert_dfs_stack(root: Optional[TreeNode]) -> Optional[TreeNode]:
    stack = [root] if root else []
    order = []
    while stack:
        node = stack.pop()
        order.append(node.val)
        node.left, node.right = node.right, node.left
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    invert_dfs_stack.last_order = order        # lưu thứ tự thăm để quan sát
    return root

t = invert_dfs_stack(build_tree([4, 2, 7, 1, 3, 6, 9]))
print("Kết quả:", to_level_list(t), "| thứ tự thăm:", invert_dfs_stack.last_order)

Kết quả: [4, 7, 2, 9, 6, 3, 1] | thứ tự thăm: [4, 2, 1, 3, 7, 6, 9]


In [13]:
import sys
deep_vals = [1]
for v in range(2, 3001):                        # cây lệch trái 3000 node
    deep_vals += [v, None]
deep = build_tree(deep_vals)
print("Chiều cao cây lệch (đo bằng BFS):", end=" ")
h, q = 0, deque([deep])
while q:
    h += 1
    for _ in range(len(q)):
        n = q.popleft()
        q.extend(c for c in (n.left, n.right) if c)
print(h, "| giới hạn đệ quy:", sys.getrecursionlimit())
try:
    invert_recursive(deep)
    print("Đệ quy: chạy được")
except RecursionError:
    print("Đệ quy: RecursionError (quá sâu)")
deep = build_tree(deep_vals)
invert_dfs_stack(deep)
print("Stack lặp: chạy được, root.right.val =", deep.right.val)

Chiều cao cây lệch (đo bằng BFS): 3000 | giới hạn đệ quy: 1000
Đệ quy: RecursionError (quá sâu)
Stack lặp: chạy được, root.right.val = 2


Với constraints của đề (≤ 100 node) đệ quy hoàn toàn an toàn; thí nghiệm trên chỉ để hiểu giới hạn khi gặp cây lớn ở bài khác.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def invertTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        if root is None:
            return None
        root.left, root.right = root.right, root.left
        self.invertTree(root.left)
        self.invertTree(root.right)
        return root

print(to_level_list(Solution().invertTree(build_tree([4, 2, 7, 1, 3, 6, 9]))))

[4, 7, 2, 9, 6, 3, 1]


Bản BFS (không đệ quy) cũng nộp được:

```python
from collections import deque

class Solution:
    def invertTree(self, root):
        if not root:
            return None
        q = deque([root])
        while q:
            node = q.popleft()
            node.left, node.right = node.right, node.left
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        return root
```

## 10. Vì sao đúng? (quy nạp theo chiều cao)

Gọi `mirror(T)` là ảnh gương của cây `T`. Định nghĩa: `mirror(None) = None`; với cây có gốc `r`, `mirror` có gốc `r`, cây con trái là `mirror(R)` và cây con phải là `mirror(L)` (L, R là hai cây con ban đầu).

**Khẳng định:** `invert(T)` biến `T` thành `mirror(T)`.
- **Cơ sở** (chiều cao 0, `T = None`): trả `None` = `mirror(None)`.
- **Bước quy nạp:** giả sử đúng với mọi cây chiều cao `< h`. Với cây chiều cao `h`: sau khi đổi chỗ, `root.left = R`, `root.right = L`. Hai cây con có chiều cao `< h` nên theo giả thiết, `invert(R)` cho `mirror(R)` và `invert(L)` cho `mirror(L)`. Vậy root có con trái `mirror(R)`, con phải `mirror(L)` — đúng định nghĩa.

**BFS/stack:** mỗi node được lấy ra đúng một lần và đổi hai con đúng một lần; việc đổi ở một node không làm thay đổi tập node (chỉ đổi vị trí), nên cuối cùng mọi node đều đã đổi — tương đương định nghĩa ảnh gương.

## 11. Time / Space — so sánh

Gọi `n` = số node, `h` = chiều cao (`log2 n` ≈ `h` khi cây cân bằng; `h = n` khi cây lệch).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Dựng cây gương mới | `O(n)` | `O(n)` node mới + `O(h)` stack | không sửa cây gốc |
| 2. Đệ quy tại chỗ | `O(n)` | `O(h)` call stack | **ngắn, chuẩn** |
| 3. BFS queue | `O(n)` | `O(w)` — `w` là số node lớn nhất trên một tầng (≤ `n/2`+1) | không đệ quy |
| 4. DFS stack | `O(n)` | `O(h)` | không đệ quy |

Mỗi node được thăm đúng một lần với `O(1)` công việc (một phép đổi chỗ).

## 12. Bộ test

Ngoài so sánh level order, ta kiểm tra thêm **tính chất**: lật hai lần phải quay về cây ban đầu; lật cây đối xứng thì không đổi; thứ tự **inorder** (trái–gốc–phải) của cây lật = đảo ngược inorder của cây gốc.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
inv = lambda vals: to_level_list(sol.invertTree(build_tree(vals)))
check("ví dụ 1", inv([4, 2, 7, 1, 3, 6, 9]), [4, 7, 2, 9, 6, 3, 1])
check("ví dụ 2", inv([2, 1, 3]), [2, 3, 1])
check("ví dụ 3 rỗng", inv([]), [])
check("một node", inv([5]), [5])
check("chỉ con trái", inv([1, 2]), [1, None, 2])
check("chỉ con phải", inv([1, None, 2]), [1, 2])
check("cây lệch", inv([1, 2, None, 3]), [1, None, 2, None, 3])
check("cây đối xứng không đổi", inv([1, 2, 2, 3, 4, 4, 3]), [1, 2, 2, 3, 4, 4, 3])
check("giá trị âm", inv([-1, -2, -3]), [-1, -3, -2])

PASS ví dụ 1: [4, 7, 2, 9, 6, 3, 1]
PASS ví dụ 2: [2, 3, 1]
PASS ví dụ 3 rỗng: []
PASS một node: [5]
PASS chỉ con trái: [1, None, 2]
PASS chỉ con phải: [1, 2]
PASS cây lệch: [1, None, 2, None, 3]
PASS cây đối xứng không đổi: [1, 2, 2, 3, 4, 4, 3]
PASS giá trị âm: [-1, -3, -2]


In [16]:
import random

def inorder(node, out=None):
    out = [] if out is None else out
    if node:
        inorder(node.left, out); out.append(node.val); inorder(node.right, out)
    return out

def random_tree(rng, n):
    # dựng cây ngẫu nhiên n node bằng cách chèn lần lượt vào vị trí trống ngẫu nhiên
    if n == 0:
        return None
    root = TreeNode(rng.randint(-100, 100))
    nodes = [root]
    for _ in range(n - 1):
        while True:
            p = rng.choice(nodes)
            side = rng.choice(["left", "right"])
            if getattr(p, side) is None:
                child = TreeNode(rng.randint(-100, 100))
                setattr(p, side, child)
                nodes.append(child)
                break
    return root

rng = random.Random(226)
methods = [invert_recursive, invert_postorder, invert_bfs, invert_dfs_stack, Solution().invertTree]
for _ in range(500):
    n = rng.randint(0, 30)
    seed = rng.random()
    expected = to_level_list(mirror_copy(random_tree(random.Random(seed), n)))
    for m in methods:
        t = random_tree(random.Random(seed), n)            # cùng seed -> cùng cây
        before_in = inorder(t)
        got = m(t)
        assert to_level_list(got) == expected, m
        assert inorder(got) == before_in[::-1]             # tính chất inorder đảo ngược
        assert to_level_list(m(got)) == to_level_list(random_tree(random.Random(seed), n))  # lật 2 lần
print("5 cài đặt khớp cây gương trên 500 cây ngẫu nhiên + 2 tính chất")

5 cài đặt khớp cây gương trên 500 cây ngẫu nhiên + 2 tính chất


`getattr(p, "left")` đọc thuộc tính theo tên chuỗi; `setattr(p, "left", child)` gán thuộc tính theo tên chuỗi — tiện khi chọn ngẫu nhiên trái/phải.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đổi chỗ bằng hai lệnh gán riêng
```python
root.left = root.right
root.right = root.left     # lúc này root.left đã là right cũ -> hai con giống nhau!
```

In [17]:
def invert_bug_two_assign(root):
    if root is None:
        return None
    root.left = root.right
    root.right = root.left
    invert_bug_two_assign(root.left)
    invert_bug_two_assign(root.right)
    return root

print("Bản lỗi [2,1,3]:", to_level_list(invert_bug_two_assign(build_tree([2, 1, 3]))), "(đúng phải là [2, 3, 1])")

Bản lỗi [2,1,3]: [2, 3, 3] (đúng phải là [2, 3, 1])


Muốn viết hai dòng thì cần biến tạm: `tmp = root.left; root.left = root.right; root.right = tmp`. Trong Python, `a, b = b, a` gọn và an toàn hơn.

### Lỗi 2 — Chỉ đổi ở root (quên đệ quy)
Đã thấy ở mục 3: `[4, 7, 2, 6, 9, 1, 3]` thay vì `[4, 7, 2, 9, 6, 3, 1]`.

### Lỗi 3 — Bản postorder nhưng gán không chéo
```python
root.left, root.right = invert(root.left), invert(root.right)    # SAI: lật bên trong nhưng không đổi vị trí
```

In [18]:
def invert_bug_not_crossed(root):
    if root is None:
        return None
    root.left, root.right = invert_bug_not_crossed(root.left), invert_bug_not_crossed(root.right)
    return root

print("Bản lỗi không chéo:", to_level_list(invert_bug_not_crossed(build_tree([4, 2, 7, 1, 3, 6, 9]))),
      "(cây không đổi!)")

Bản lỗi không chéo: [4, 2, 7, 1, 3, 6, 9] (cây không đổi!)


### Lỗi 4 — Quên trường hợp cơ sở `None`
Thiếu `if root is None: return None` → gọi `root.left` trên `None` → `AttributeError: 'NoneType' object has no attribute 'left'`.

### Lỗi 5 — Quên `return root`
Hàm Python không có `return` sẽ trả `None`. LeetCode đọc giá trị trả về → in ra cây rỗng `[]`.

In [19]:
def invert_bug_no_return(root):
    if root is None:
        return None
    root.left, root.right = root.right, root.left
    invert_bug_no_return(root.left)
    invert_bug_no_return(root.right)
    # quên return root

print("Bản quên return:", to_level_list(invert_bug_no_return(build_tree([2, 1, 3]))), "(đúng phải là [2, 3, 1])")

Bản quên return: [] (đúng phải là [2, 3, 1])


## 14. Mẹo nhận dạng pattern

- Bài cây hỏi một phép biến đổi/thuộc tính **áp dụng giống nhau ở mọi node** → **đệ quy**: "làm ở node hiện tại + giao cho hai cây con".
- Mẫu tư duy 3 câu hỏi cho đệ quy trên cây:
  1. Cây rỗng thì trả gì?
  2. Ở node hiện tại phải làm gì?
  3. Kết quả của hai cây con được dùng thế nào?
- Muốn tránh đệ quy → BFS (queue) hoặc DFS (stack) "thăm mọi node một lần".
- Bài liên quan: 101 Symmetric Tree (so cây với ảnh gương của chính nó), 100 Same Tree, 104 Maximum Depth, 951 Flip Equivalent Binary Trees, 617 Merge Two Binary Trees.

## 15. Biến thể

### 15.1. Symmetric Tree (LeetCode 101) dùng lại ý tưởng gương
Cây đối xứng ⇔ cây con trái là ảnh gương của cây con phải. Viết `is_mirror(a, b)`: cả hai rỗng → `True`; một rỗng → `False`; còn lại cần `a.val == b.val`, `is_mirror(a.left, b.right)`, `is_mirror(a.right, b.left)`.

### 15.2. Chỉ lật các tầng lẻ (Reverse Odd Levels — ý tưởng)
Duyệt BFS theo tầng, ở tầng lẻ đảo **giá trị** của các node trong tầng.

In [20]:
def is_mirror(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return a.val == b.val and is_mirror(a.left, b.right) and is_mirror(a.right, b.left)

def is_symmetric(root):
    return root is None or is_mirror(root.left, root.right)

print("101 [1,2,2,3,4,4,3]:", is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))
print("101 [1,2,2,None,3,None,3]:", is_symmetric(build_tree([1, 2, 2, None, 3, None, 3])))

def reverse_odd_levels(root):
    level, q = 0, deque([root]) if root else deque()
    while q:
        nodes = list(q)
        if level % 2 == 1:
            vals = [n.val for n in nodes][::-1]
            for n, v in zip(nodes, vals):
                n.val = v
        q = deque(c for n in nodes for c in (n.left, n.right) if c)
        level += 1
    return root

print("Lật giá trị tầng lẻ [2,3,5,8,13,21,34]:", to_level_list(reverse_odd_levels(build_tree([2, 3, 5, 8, 13, 21, 34]))))

101 [1,2,2,3,4,4,3]: True
101 [1,2,2,None,3,None,3]: False
Lật giá trị tầng lẻ [2,3,5,8,13,21,34]: [2, 5, 3, 8, 13, 21, 34]


## 16. Bài tập tự luyện

### Bài 1 — Vẽ tay
Vẽ cây `[1, 2, 3, 4, None, None, 5]` và cây sau khi lật. Viết level order của kết quả.

### Bài 2 — Dự đoán
Với cách BFS, nếu đổi `queue.popleft()` thành `queue.pop()` (lấy ở cuối), kết quả cuối cùng có còn đúng không? Vì sao?

### Bài 3 — TODO: tự viết bản đệ quy

In [21]:
def exercise_invert(root):
    # TODO 1: cơ sở None
    # TODO 2: đổi chỗ hai con
    # TODO 3: lật hai cây con, trả root
    pass

# assert to_level_list(exercise_invert(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [4, 7, 2, 9, 6, 3, 1]
# assert exercise_invert(None) is None

### Bài 4 — TODO: tự viết bản BFS

In [22]:
def exercise_invert_bfs(root):
    # TODO: deque([root]); popleft; đổi; append các con khác None
    pass

# assert to_level_list(exercise_invert_bfs(build_tree([2, 1, 3]))) == [2, 3, 1]

### Bài 5 — Biến thể: kiểm tra hai cây có phải ảnh gương của nhau không
`are_mirrors(build_tree([1,2,3]), build_tree([1,3,2]))` → `True`.

In [23]:
def exercise_are_mirrors(a, b):
    # TODO: không được sửa cây; dùng đệ quy như is_mirror
    pass

# assert exercise_are_mirrors(build_tree([1, 2, 3]), build_tree([1, 3, 2])) is True
# assert exercise_are_mirrors(build_tree([1, 2]), build_tree([1, 2])) is False

### Bài 6 — Mini challenge: Flip Equivalent (LeetCode 951)
Hai cây "lật tương đương" nếu có thể biến cây này thành cây kia bằng cách đổi hai con ở **một số** node (không nhất thiết mọi node).
`flip_equiv([1,2,3,4,5,6,None,None,None,7,8], [1,3,2,None,6,4,5,None,None,None,None,8,7])` → `True`.

In [24]:
def exercise_flip_equiv(a, b):
    # TODO: cùng rỗng / khác val / (không lật và tương đương) hoặc (lật và tương đương)
    pass

# assert exercise_flip_equiv(build_tree([1,2,3,4,5,6,None,None,None,7,8]),
#                            build_tree([1,3,2,None,6,4,5,None,None,None,None,8,7])) is True
# assert exercise_flip_equiv(build_tree([1, 2]), build_tree([1, None, 3])) is False

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đổi con ở mọi node | 1 có con 2 (trái), 3 (phải); 2 có con trái 4; 3 có con phải 5 | kết quả: 1 có (3, 2); 3 có con trái 5; 2 có con phải 4 |
| 2 | Thứ tự thăm có quan trọng không? | mỗi node vẫn bị đổi đúng 1 lần | vẫn đúng (thành DFS) |
| 3 | 3 câu hỏi đệ quy | `root.left, root.right = root.right, root.left` | gọi cho hai con rồi `return root` |
| 4 | Queue thăm mọi node | `node = q.popleft()` | `if node.left: q.append(node.left)` |
| 5 | Gương: trái của a ~ phải của b | hai `None` → True; một `None` → False | `a.val == b.val and f(a.left, b.right) and f(a.right, b.left)` |
| 6 | Mỗi node có 2 lựa chọn | thử cả "không lật" và "lật" | `(f(a.l,b.l) and f(a.r,b.r)) or (f(a.l,b.r) and f(a.r,b.l))` |

## Đáp án tham khảo

**Bài 1:** kết quả level order `[1, 3, 2, 5, None, None, 4]`.

```text
   1              1
  / \            / \
 2   3   ->     3   2
/     \        /     \
4      5      5       4
```

**Bài 2:** Vẫn đúng. `pop()` biến queue thành stack → thứ tự thăm theo chiều sâu, nhưng mỗi node vẫn được lấy ra và đổi hai con đúng một lần.

In [25]:
def answer_invert(root):
    if root is None:
        return None
    root.left, root.right = root.right, root.left
    answer_invert(root.left)
    answer_invert(root.right)
    return root

def answer_invert_bfs(root):
    if root is None:
        return None
    q = deque([root])
    while q:
        node = q.popleft()
        node.left, node.right = node.right, node.left
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return root

def answer_are_mirrors(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    return answer_are_mirrors(a.left, b.right) and answer_are_mirrors(a.right, b.left)

def answer_flip_equiv(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    no_flip = answer_flip_equiv(a.left, b.left) and answer_flip_equiv(a.right, b.right)
    flip = answer_flip_equiv(a.left, b.right) and answer_flip_equiv(a.right, b.left)
    return no_flip or flip

def invert_bfs_with_pop(root):
    q = deque([root]) if root else deque()
    while q:
        node = q.pop()
        node.left, node.right = node.right, node.left
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return root

assert to_level_list(answer_invert(build_tree([1, 2, 3, 4, None, None, 5]))) == [1, 3, 2, 5, None, None, 4]
assert to_level_list(invert_bfs_with_pop(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [4, 7, 2, 9, 6, 3, 1]
assert to_level_list(answer_invert(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [4, 7, 2, 9, 6, 3, 1]
assert answer_invert(None) is None
assert to_level_list(answer_invert_bfs(build_tree([2, 1, 3]))) == [2, 3, 1]
assert answer_are_mirrors(build_tree([1, 2, 3]), build_tree([1, 3, 2])) is True
assert answer_are_mirrors(build_tree([1, 2]), build_tree([1, 2])) is False
assert answer_flip_equiv(build_tree([1, 2, 3, 4, 5, 6, None, None, None, 7, 8]),
                         build_tree([1, 3, 2, None, 6, 4, 5, None, None, None, None, 8, 7])) is True
assert answer_flip_equiv(build_tree([1, 2]), build_tree([1, None, 3])) is False
print("Đáp án tham khảo PASS toàn bộ!")

Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
def invertTree(root):
    if root is None:                                   # 1. cây rỗng
        return None
    root.left, root.right = root.right, root.left      # 2. việc ở node hiện tại
    invertTree(root.left)                              # 3. giao cho hai cây con
    invertTree(root.right)
    return root
```

Khuôn đệ quy cây: **cơ sở → việc ở node → gọi hai con → kết hợp/trả về**.

**Checklist:**
- [ ] Có trường hợp `None`.
- [ ] Đổi chỗ bằng `a, b = b, a` (hoặc biến tạm), không hai lệnh gán rời.
- [ ] Đổi ở **mọi** node, không chỉ root.
- [ ] Có `return root`.
- [ ] Đã thử cây rỗng, 1 node, chỉ con trái/phải, cây lệch.
- [ ] Nói được Time `O(n)`, Space `O(h)`.